# Timelines, Events, and Maps

Load real musical data, explore it as an `EventStore`, create a
`Timeline`, and attach a `ConversionMap` to translate between units.

In [1]:
from pathlib import Path

from timetoalign.loader.score.partitura import PartituraLoader
from timetoalign.maps import TicksToQuarters

## Load Events from a Score

In [2]:
DATA_DIR = Path(".").resolve().parents[1] / "tests" / "data" / "vienna_1x22"

loader = PartituraLoader()
loader.load(DATA_DIR / "Chopin_op10_no3.musicxml")
loader.store

AttributeError: 'int' object has no attribute 'get'

ScoreStore(notes=498, measures=22, controls=22, annotations=5)

## Create a Timeline

In [3]:
tl = loader.create_timeline(uid="chopin_etude")
tl

ContinuousLogicalTimeline(id='chopin_etude', length=41.5, unit=quarters, events=0, children=4, cmaps=3)

## Access Events

The loader creates child timelines for each event category (notes,
measures, controls). Access them via `get_child()`.

In [4]:
tl.get_child("notes").get_events(event_type="Note").to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,duration_float,mc,mn,...,spelled_pitch,tpc,octave,velocity,tied,gracenote,chord_id,voice,staff,part_id
0,notes:note:000001,B3,interval,Note,0,0.50,1/2,0.50,1,1,...,"{'gpc_int': 6, 'gpc_str': 'B', 'acc': 0, 'spc_...",5,3,64,0,NaN,NaN,1,1,P1
1,notes:note:000002,E4,interval,Note,1/2,1.00,1/2,0.50,2,2,...,"{'gpc_int': 2, 'gpc_str': 'E', 'acc': 0, 'spc_...",4,4,64,0,NaN,NaN,1,1,P1
2,notes:note:000003,G♯3,interval,Note,1/2,0.75,1/4,0.25,2,2,...,"{'gpc_int': 4, 'gpc_str': 'G', 'acc': 1, 'spc_...",8,3,64,0,NaN,NaN,3,1,P1
3,notes:note:000004,E2,interval,Note,1/2,0.75,1/4,0.25,2,2,...,"{'gpc_int': 2, 'gpc_str': 'E', 'acc': 0, 'spc_...",4,2,64,0,NaN,NaN,4,2,P1
4,notes:note:000005,E2,interval,Note,1/2,1.50,1,1.00,2,2,...,"{'gpc_int': 2, 'gpc_str': 'E', 'acc': 0, 'spc_...",4,2,64,0,NaN,NaN,7,2,P1


## Coordinates

A `Coordinate` binds a number to a unit, ensuring type-safe arithmetic.

In [5]:
coord = tl.make_coordinate(8)
coord

Coordinate(8, quarters)

## Conversion Maps (C-Maps)

Attach a map to translate the timeline's native `quarters` into MIDI
`ticks` (at 480 pulses per quarter).

In [6]:
q2t = TicksToQuarters(ppq=480).inverse()
tl.add_conversion_map(q2t)

ticks_coord = tl.convert_to(coord, target_unit="ticks")
{
    "input": f"{coord.value} {coord.unit.name}",
    "output": f"{ticks_coord}",
}

{'input': '8 quarters', 'output': '3840 ticks'}

**Next:** [Children, Regions & Timestamps](tut01b_children_regions_timestamps.ipynb)